# Prepare contigs for nf-damage inference
Lucia Winkler, 28.07.2025, 15.10.2025

based on create_fasta_contig_genus.ipynb and contigs_taxo.ipynb by Maxime Borry
## What this notebook does:
Input data: contig taxonomic annotation, unfiltered pydamage output from nf-core/MAG run

The notebook filters for genera that are more abundant in 95% of the TDM datasets than in extraction blanks, based on proportion of mapped reads

The notebook filters for contigs with min. 50x coverage

**selection of core genera:**

The notebook selects genera, that appear in >90% (min.55/60 samples) of the stalagmite sequence datasets after filtering 

The sequences of the selected contigs are extracted from assembly fastas and put into pseudo-bins ordered by genus and sample

Sample sheets for the nf-damage inference pipeline are generated.

**Selection of test genera**

The notebook selects genera, that appear in 50-54/60 of the stalagmite sequence datasets after filtering 

The sequences of the selected contigs are extracted from assembly fastas and put into pseudo-bins ordered by genus and sample

Sample sheets for the nf-damage inference pipeline are generated.

In [1]:
import pandas as pd
import os
import glob
import taxopy
import re
from pathlib import Path
#from scipy.stats import mannwhitneyu
from collections import defaultdict
from tqdm import tqdm
import pysam


### Functions

In [ ]:
"Read and concatenate contig taxonomic annotation"
# Configuration
DATA_PATH = '<pat to>/MMSeqs_Contig_Taxonomy'
TSV_PATTERN = os.path.join(DATA_PATH, '*.tsv')

# Columns of interest
COLUMN_MAP = {
    0: 'contig',
    1: 'taxID',
    2: 'rank',
    3: 'taxon '
}

def load_and_process_tsv(file_path):
    sample_name = os.path.splitext(os.path.basename(file_path))[0]
    df = pd.read_csv(file_path, sep='\t', header=None)
    df = df.assign(sample=sample_name)

    # Rename selected columns
    df = df.rename(columns=COLUMN_MAP)

    # Drop columns 0 to 8 (inclusive), except renamed ones
    keep_cols = list(COLUMN_MAP.values()) + ['sample']
    df = df[keep_cols]
    return df


In [3]:
"Add genus and species name to tax. annotation"
taxdb = taxopy.TaxDb()

def get_genus(taxid, taxdb=taxdb):
    try:
        _ = taxopy.Taxon(taxid, taxdb)
        if "genus" in _.rank_name_dictionary:
            g =  _.rank_name_dictionary['genus']
        else:
            g = None
        if "species" in _.rank_name_dictionary:
            s =  _.rank_name_dictionary['species']
        else:
            s = None
    except taxopy.exceptions.TaxidError:
        g = None
        s = None
    return g, s

In [ ]:
'read unfiltered pydamage results table'
# Configuration
BASE_DIR = Path('<path to>/PyDamage_Full/analyze')
RESULT_FILENAME = 'pydamage_results/pydamage_results.csv'

# Patterns to support both TDM and EXB formats
ASSEMBLY_REGEX = r"(TDM\d{3}|EXB\d{3}_A\d{4})"

def find_pydamage_files(base_dir: Path, result_filename: str):
    """Finds all pydamage result files in subfolders."""
    folders = [f for f in base_dir.iterdir() if f.is_dir()]
    result_paths = [(f.name, f / result_filename) for f in folders if (f / result_filename).exists()]
    return result_paths

def load_and_annotate(file_info):
    """Loads a CSV and adds 'sample' column."""
    name, path = file_info
    df = pd.read_csv(path, usecols=range(16))
    df['sample'] = name
    df = df.rename(columns={'reference': 'contig'})
    return df

def standardize_assembly_name(name):
    """Extracts TDMxxx or EXB080_Axxxx pattern."""
    match = re.search(ASSEMBLY_REGEX, name)
    return match.group(0) if match else name  # fallback to original name if no match


In [5]:
'Extract core contigs into fasta'
def create_fasta(df, sample, genus, in_fa, outdir):
    genus_out = genus.lower().replace(" ", "_")
    out_fa = os.path.join(outdir, f"{sample}_{genus_out}.fa")
    # print(genus, sample)
    contigs = df[(df['genus'] == genus) & (df['sample'] == sample)].contig.values
    # print(contigs)
    if len(contigs) > 0:
        with pysam.FastxFile(in_fa) as fin, open(out_fa, "w") as fout:
            for entry in fin:
                if entry.name in contigs:
                    fout.write(str(entry) + "\n")

### Load data

In [6]:
# Process all taxonomic annotation TSV files and concatenate into df
all_dfs = [load_and_process_tsv(f) for f in glob.glob(TSV_PATTERN)]
tax_df = pd.concat(all_dfs, ignore_index=True)


In [7]:
# add genus and species name to df
tax_df['genus'], tax_df['species'] =  zip(*tax_df['taxID'].map(get_genus))

In [8]:
# Load pydamage data
# Step 1: Locate files
file_list = find_pydamage_files(BASE_DIR, RESULT_FILENAME)

# Step 2: Load and annotate
dataframes = [load_and_annotate(info) for info in file_list]

# Step 3: Concatenate into single DataFrame
pydamage_df = pd.concat(dataframes, ignore_index=True)

# Step 4: Standardize assembly names
pydamage_df['sample'] = pydamage_df['sample'].apply(standardize_assembly_name)

### Quality control

In [9]:
pydamage_df.head()
pydamage_df['sample'].unique()

array(['EXB080_A0101', 'TDM033', 'TDM042', 'EXB080_A0901', 'EXB080_A0801',
       'TDM056', 'TDM030', 'TDM001', 'TDM038', 'TDM051', 'TDM017',
       'TDM019', 'TDM055', 'TDM036', 'TDM003', 'TDM037', 'TDM027',
       'EXB080_A1101', 'TDM013', 'TDM053', 'TDM052', 'TDM060', 'TDM049',
       'TDM005', 'TDM043', 'TDM014', 'TDM023', 'TDM015', 'TDM054',
       'TDM024', 'EXB080_A1001', 'TDM007', 'TDM034', 'TDM016', 'TDM008',
       'TDM012', 'EXB080_A1201', 'TDM022', 'TDM035', 'TDM006', 'TDM057',
       'TDM010', 'TDM045', 'TDM028', 'TDM025', 'TDM046', 'TDM020',
       'TDM061', 'EXB080_A0102', 'TDM011', 'TDM039', 'TDM031', 'TDM002',
       'TDM029', 'TDM004', 'TDM021', 'EXB080_A1401', 'EXB080_A1301',
       'TDM040', 'TDM044', 'TDM048', 'TDM018', 'TDM026', 'TDM041',
       'TDM032', 'TDM059', 'TDM047', 'TDM050', 'TDM058'], dtype=object)

In [10]:

tax_df['sample'].unique()

array(['TDM056', 'TDM025', 'TDM053', 'TDM061', 'TDM051', 'TDM019',
       'EXB080_A1101', 'TDM029', 'EXB080_A0801', 'TDM010', 'TDM016',
       'TDM054', 'TDM001', 'TDM038', 'TDM052', 'TDM020', 'TDM013',
       'TDM055', 'TDM044', 'EXB080_A1001', 'TDM050', 'TDM031', 'TDM039',
       'EXB080_A0901', 'TDM046', 'EXB080_A0101', 'TDM006', 'TDM060',
       'TDM002', 'TDM030', 'TDM035', 'TDM058', 'EXB080_A1201', 'TDM015',
       'TDM005', 'TDM018', 'TDM017', 'TDM059', 'TDM003', 'TDM012',
       'TDM008', 'EXB080_A1401', 'TDM022', 'TDM028', 'TDM027', 'TDM026',
       'TDM049', 'TDM045', 'TDM048', 'TDM036', 'TDM021', 'TDM037',
       'TDM033', 'TDM047', 'EXB080_A1301', 'EXB080_A0102', 'TDM023',
       'TDM024', 'TDM004', 'TDM043', 'TDM011', 'TDM041', 'TDM032',
       'TDM014', 'TDM042', 'TDM057', 'TDM007', 'TDM034', 'TDM040'],
      dtype=object)

In [11]:
pd.options.display.max_rows = 4000
contignr=pydamage_df.groupby(['sample'])['contig'].count()
print(contignr)

sample
EXB080_A0101       500
EXB080_A0102       658
EXB080_A0801     13181
EXB080_A0901      7063
EXB080_A1001      4341
EXB080_A1101      6574
EXB080_A1201      6078
EXB080_A1301     10032
EXB080_A1401      3975
TDM001          101739
TDM002           80131
TDM003           65595
TDM004           53669
TDM005           45573
TDM006           31067
TDM007           26105
TDM008           32102
TDM010           69972
TDM011           48712
TDM012           17425
TDM013           54751
TDM014           57608
TDM015           42869
TDM016           48549
TDM017           55289
TDM018           49128
TDM019           50802
TDM020           34657
TDM021           60583
TDM022           28772
TDM023           46897
TDM024           51687
TDM025           24599
TDM026           24853
TDM027           48475
TDM028           35208
TDM029           49519
TDM030           41507
TDM031           66849
TDM032           33565
TDM033           41468
TDM034           53078
TDM035           51742
TDM0

In [12]:
pd.options.display.max_rows = 4000
assemblylength=pydamage_df.groupby(['sample'])['reflen'].sum()
print(assemblylength)

sample
EXB080_A0101       126525
EXB080_A0102       170283
EXB080_A0801      5019349
EXB080_A0901      2530167
EXB080_A1001      1747513
EXB080_A1101      2341815
EXB080_A1201      2769890
EXB080_A1301      3682717
EXB080_A1401      2028478
TDM001          106107486
TDM002           84817480
TDM003           81330653
TDM004           57818381
TDM005           56296858
TDM006           39358294
TDM007           40088542
TDM008           36833252
TDM010           69340663
TDM011           51012858
TDM012           20165542
TDM013           51594039
TDM014           58499409
TDM015           37718055
TDM016           49864394
TDM017           48044553
TDM018           45547982
TDM019           56820283
TDM020           37017429
TDM021           61538095
TDM022           28061196
TDM023           38223225
TDM024           38034926
TDM025           22934633
TDM026           20460615
TDM027           34648094
TDM028           28792812
TDM029           39109896
TDM030           37676907
TDM03

In [13]:
print("contigs in pydamage output:", len(pydamage_df), "\ncontigs in tax. annotation: ", len(tax_df))

contigs in pydamage output: 2302803 
contigs in tax. annotation:  2073597


Not all contigs were taxonomically annotated.

### Merge data

In [14]:
full_df = pydamage_df.merge(tax_df, on=["sample", "contig"] , how="left")
len(full_df)

2302803

### Calculating coverages in TPM
I'm aiming to use this to quantify, which genera are more abundant in Extraction blanks EXB than in stalagmite samples (TDM)

In [15]:
 # Step 1: Calculate RPK for each contig
full_df["rpk"] = full_df["nb_reads_aligned"] / (full_df["reflen"] / 1000)

# Step 2: Calculate TPM per sample
# Group by sample, calculate per-sample sum of RPKs
full_df["rpk_sum"] = full_df.groupby("sample")["rpk"].transform("sum")

# Step 3: Compute TPM
full_df["tpm"] = (full_df["rpk"] / full_df["rpk_sum"]) * 1e6

In [16]:
full_df.groupby(['sample'])['tpm'].sum()

sample
EXB080_A0101    1000000.0
EXB080_A0102    1000000.0
EXB080_A0801    1000000.0
EXB080_A0901    1000000.0
EXB080_A1001    1000000.0
EXB080_A1101    1000000.0
EXB080_A1201    1000000.0
EXB080_A1301    1000000.0
EXB080_A1401    1000000.0
TDM001          1000000.0
TDM002          1000000.0
TDM003          1000000.0
TDM004          1000000.0
TDM005          1000000.0
TDM006          1000000.0
TDM007          1000000.0
TDM008          1000000.0
TDM010          1000000.0
TDM011          1000000.0
TDM012          1000000.0
TDM013          1000000.0
TDM014          1000000.0
TDM015          1000000.0
TDM016          1000000.0
TDM017          1000000.0
TDM018          1000000.0
TDM019          1000000.0
TDM020          1000000.0
TDM021          1000000.0
TDM022          1000000.0
TDM023          1000000.0
TDM024          1000000.0
TDM025          1000000.0
TDM026          1000000.0
TDM027          1000000.0
TDM028          1000000.0
TDM029          1000000.0
TDM030          1000000.0
TDM03

### Identify genera that are robustly more abundant in TDM samples compared to EXB

Selection was done based on TPM coverage values. TPM values of contigs assigned to the same genus were summed up (cumulative TPM). I then filtered for genera, that hat a higher cumulative TPM in >95% of the samples compared to the maximum cumulative TPM in the extraction blanks.

In [17]:
counts = (
    full_df.groupby(['sample', 'genus'])['tpm']
      .sum()
      .reset_index(name='cumulative_tpm')
)

In [18]:
counts['sample_type'] = counts['sample'].apply(lambda s: 'EXB' if s.startswith('EXB') else 'TDM')
counts

,sample,genus,cumulative_tpm,sample_type
0,EXB080_A0101,Burkholderia,323807.154961,EXB
1,EXB080_A0101,Carltongylesvirus,529.368277,EXB
2,EXB080_A0101,Culex,4958.320414,EXB
3,EXB080_A0101,Elysia,3942.647391,EXB
4,EXB080_A0101,Escherichia,3870.270289,EXB
...,...,...,...,...
26562,TDM061,Vulcaniibacterium,53.988513,TDM
26563,TDM061,Xanthomonas,64.299224,TDM
26564,TDM061,Xenopsylla,36.527844,TDM
26565,TDM061,Zavarzinia,37.583439,TDM


In [19]:
'Select genera, that (cumulatively) have more mapped reads than the maximum number of cumulative mapped reads in EXB samples in 95% of the TDM samples'
robust_genera = []

for genus, group in counts.groupby('genus'):
    exb_vals = group[group['sample_type'] == 'EXB']['cumulative_tpm']
    tdm_vals = group[group['sample_type'] == 'TDM']['cumulative_tpm']
    
    # Always keep if genus is missing in EXB
    if len(exb_vals) == 0:
        robust_genera.append(genus)
        continue

    # Apply >95% rule
    exb_max = exb_vals.max()
    tdm_above_exb = (tdm_vals > exb_max).sum()
    tdm_fraction = tdm_above_exb / 60 

    if tdm_fraction > 0.95:
        robust_genera.append(genus)
robust_genera

['Abditibacterium',
 'Abyssibacter',
 'Acanthopleuribacter',
 'Acaryochloris',
 'Acer',
 'Acerihabitans',
 'Acetobacteroides',
 'Acetomicrobium',
 'Acetonema',
 'Achromatium',
 'Acidaminococcus',
 'Acidibrevibacterium',
 'Acidicapsa',
 'Acidiferrimicrobium',
 'Acidiferrobacter',
 'Acidihalobacter',
 'Acidimicrobium',
 'Acidiphilium',
 'Acidipila',
 'Acidipropionibacterium',
 'Acidisoma',
 'Aciditerrimonas',
 'Acidithiobacillus',
 'Acidithrix',
 'Acidobacterium',
 'Acidocella',
 'Acidothermus',
 'Acorus',
 'Acrobeloides',
 'Acrocarpospora',
 'Acromyrmex',
 'Acropora',
 'Actibacterium',
 'Actimicrobium',
 'Actinacidiphila',
 'Actinoallomurus',
 'Actinobacillus',
 'Actinocatenispora',
 'Actinocorallia',
 'Actinocrinis',
 'Actinokineospora',
 'Actinomadura',
 'Actinomarinicola',
 'Actinomortierella',
 'Actinomyces',
 'Actinophytocola',
 'Actinopolymorpha',
 'Actinopolyspora',
 'Actinospica',
 'Actinosynnema',
 'Actinotalea',
 'Actomonas',
 'Acutalibacter',
 'Acuticoccus',
 'Adhaeribacter',

### Exclude contigs from EXB-dominant genera and contigs with <50x coverage

In [20]:
# Step 2: Filter out rows where genus is in that set
filtered_df = full_df[full_df['genus'].isin(robust_genera) &
    (full_df['coverage'] >= 50)]

### Check for continuous presence over time
I selected genera, that - after all filtering steps - appeared >90% of the stalagmite samples

In [21]:
wide_df = (
    filtered_df.groupby(['sample', 'genus'])['contig']
      .count()
      .reset_index(name='n_contigs')  # Step 1: count contigs
      .pivot_table(index='sample', columns='genus', values='n_contigs')  # Step 2: pivot
      .reset_index()  # optional: make 'sample' a column instead of index
)

wide_df = wide_df[~wide_df['sample'].str.startswith('EXB')]
wide_df

genus,sample,Abditibacterium,Abyssibacter,Acanthopleuribacter,Acaryochloris,Acetomicrobium,Achromatium,Acidaminococcus,Acidicapsa,Acidiferrimicrobium,...,Zavarzinia,Zea,Zeimonas,Zemynaea,Zestomonas,Zoogloea,Zooshikella,Zopfia,Zwartia,Zymobacter
2,TDM001,NaN,4.0,NaN,1.0,NaN,NaN,NaN,NaN,2.0,...,NaN,1.0,3.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN
3,TDM002,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,3.0,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,TDM003,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN
5,TDM004,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN
6,TDM005,NaN,3.0,NaN,NaN,NaN,NaN,NaN,1.0,2.0,...,NaN,NaN,1.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN
7,TDM006,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN
8,TDM007,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,TDM008,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN
10,TDM010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,1.0,2.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN
11,TDM011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Selection of core genera

In [22]:
# Count NaN values in each genus (i.e., each column except 'sample')
na_counts = wide_df.isna().sum()
# Exclude 'sample' column 
na_counts = na_counts.drop('sample', errors='ignore')
# Find genera (column names) with 6 or fewer NaN values
core_genera = na_counts[na_counts < 6].index.tolist()

core_genera


['Candidatus Methylomirabilis',
 'Candidatus Nitrosotenuis',
 'Conexibacter',
 'Nitrosotalea',
 'Nitrospira',
 'Piscinibacter']

### Data frame of core genera contigs

In [23]:
core_df= filtered_df[filtered_df['genus'].isin(core_genera)]


In [24]:
core_df.groupby(['sample','genus'])['contig'].count().unstack(fill_value=0)
#qc.pivot_table(index="sample", columns="genus", values="count", fill_value=0)

genus,Candidatus Methylomirabilis,Candidatus Nitrosotenuis,Conexibacter,Nitrosotalea,Nitrospira,Piscinibacter
sample,,,,,,
EXB080_A0801,0,0,0,0,1,0
EXB080_A0901,0,0,0,0,1,0
TDM001,0,85,3,2,2333,6
TDM002,1,143,8,4,2313,4
TDM003,1,131,19,5,1945,2
TDM004,2,71,7,6,2425,2
TDM005,2,3,33,1,2331,11
TDM006,2,55,43,3,1757,2
TDM007,1,4,31,1,2016,1


### Extract core genera contigs from assembly fasta files

In [ ]:
'locate assemblies'
assemblies = Path("<path to>/Assemblies").glob("*.contigs.fa.gz")
assemblies_dict = {i.stem.split("-")[2].split(".")[0]:i.as_posix() for i in assemblies}
assemblies_dict

{'TDM032': '/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/MEGAHIT-group-TDM032.contigs.fa.gz',
 'TDM049': '/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/MEGAHIT-group-TDM049.contigs.fa.gz',
 'TDM042': '/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/MEGAHIT-group-TDM042.contigs.fa.gz',
 'TDM061': '/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/MEGAHIT-group-TDM061.contigs.fa.gz',
 'TDM028': '/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/MEGAHIT-group-TDM028.contigs.fa.gz',
 'TDM027': '/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/MEGAHIT-group-TDM027.contigs.fa.gz',
 'TDM031': '/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/MEGAHIT-group-TDM031.contigs.fa.gz',
 'EXB080_A0801': '/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/MEGAHIT-group-EXB080_A0801.contigs.fa.g

In [ ]:
outdir = "<path to>/damage_inference_core/fasta"

In [126]:
for g in tqdm(core_df.genus.unique()):
    for s in core_df['sample'].unique():
        try:
            create_fasta(core_df, s, g, assemblies_dict[s], outdir)
        except:
            print(f"Error with {g} {s}")

100%|██████████| 6/6 [04:17<00:00, 42.91s/it] 


### Prepare sample sheets for nf-damage inference pipeline

In [26]:
b = list(Path("/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/QC/").rglob("*.bam"))
b

[PosixPath('/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/QC/group-TDM039/MEGAHIT-group-TDM039-TDM039_A0301.bam'),
 PosixPath('/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/QC/group-TDM042/MEGAHIT-group-TDM042-TDM042_A0401.bam'),
 PosixPath('/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/QC/group-EXB080_A0102/MEGAHIT-group-EXB080_A0102-EXB080_A0102.bam'),
 PosixPath('/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/QC/group-TDM003/MEGAHIT-group-TDM003-TDM003_A0201.bam'),
 PosixPath('/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/QC/group-TDM011/MEGAHIT-group-TDM011-TDM011_A0301.bam'),
 PosixPath('/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/QC/group-TDM004/MEGAHIT-group-TDM004-TDM004_A0201.bam'),
 PosixPath('/mnt/archgen/users/borry/18_speleothem/results/nf-coreMAG/Assembly/MEGAHIT/QC/group-TDM014/MEGAHIT-group-TDM014-

In [ ]:
pd.DataFrame({
    "sample": [i.stem.split("-")[2] for i in b],
    "bam": [i.as_posix() for i in b]
}).to_csv("<path to>/damage_inference_core/bams.csv", index=False)

In [ ]:
f = list(Path("<path to>/damage_inference_core/fasta").glob("*.fa"))
f

[PosixPath('/home/lucia_winkler/speleothem/damage_inference_core/fasta/TDM033_nitrospira.fa'),
 PosixPath('/home/lucia_winkler/speleothem/damage_inference_core/fasta/TDM042_nitrospira.fa'),
 PosixPath('/home/lucia_winkler/speleothem/damage_inference_core/fasta/EXB080_A0901_nitrospira.fa'),
 PosixPath('/home/lucia_winkler/speleothem/damage_inference_core/fasta/EXB080_A0801_nitrospira.fa'),
 PosixPath('/home/lucia_winkler/speleothem/damage_inference_core/fasta/TDM056_nitrospira.fa'),
 PosixPath('/home/lucia_winkler/speleothem/damage_inference_core/fasta/TDM030_nitrospira.fa'),
 PosixPath('/home/lucia_winkler/speleothem/damage_inference_core/fasta/TDM001_nitrospira.fa'),
 PosixPath('/home/lucia_winkler/speleothem/damage_inference_core/fasta/TDM038_nitrospira.fa'),
 PosixPath('/home/lucia_winkler/speleothem/damage_inference_core/fasta/TDM051_nitrospira.fa'),
 PosixPath('/home/lucia_winkler/speleothem/damage_inference_core/fasta/TDM017_nitrospira.fa'),
 PosixPath('/home/lucia_winkler/speleo

In [131]:
samples = [i.stem.split("_")[0] for i in f]
genera = ["_".join(i.stem.split("_")[1:]) for i in f]
fa = [i.as_posix() for i in f]

In [ ]:
pd.DataFrame({
    "sample": samples,
    "taxon": genera,
    "genome": fa
    }).to_csv("<path to>/damage_inference_core/genomes.csv", index=False)

## Select genera for model testing
Here I focus on the genera present in 50-54/60 of the stalagmite datasets

In [28]:
test_genera = na_counts[(na_counts >= 6) & (na_counts <= 10)].index.tolist()

test_genera

['Desertibaculum',
 'Immundisolibacter',
 'Nitrosopumilus',
 'Nitrosospira',
 'Sulfuricaulis',
 'Sulfuritalea']

In [29]:
test_df= filtered_df[filtered_df['genus'].isin(test_genera)]

test_df.groupby(['sample','genus'])['contig'].count().unstack(fill_value=0)

genus,Desertibaculum,Immundisolibacter,Nitrosopumilus,Nitrosospira,Sulfuricaulis,Sulfuritalea
sample,,,,,,
TDM001,1,3,2,0,2,8
TDM002,1,2,4,0,2,4
TDM003,2,1,3,1,2,5
TDM004,2,1,8,1,2,4
TDM005,5,2,18,1,1,7
TDM006,0,2,1,1,0,1
TDM007,1,2,1,1,1,1
TDM008,2,2,1,1,2,6
TDM010,2,2,3,2,1,4


In [ ]:
outdir = "<path to>/damage_inference_model_testing/fasta"
for g in tqdm(test_df.genus.unique()):
    for s in test_df['sample'].unique():
        try:
            create_fasta(test_df, s, g, assemblies_dict[s], outdir)
        except:
            print(f"Error with {g} {s}")

100%|██████████| 6/6 [02:45<00:00, 27.63s/it]


In [ ]:
pd.DataFrame({
    "sample": [i.stem.split("-")[2] for i in b],
    "bam": [i.as_posix() for i in b]
}).to_csv("<path to>/damage_inference_model_testing/bams.csv", index=False)

In [ ]:
f = list(Path("<path to>/damage_inference_model_testing/fasta").glob("*.fa"))
f

samples = [i.stem.split("_")[0] for i in f]
genera = ["_".join(i.stem.split("_")[1:]) for i in f]
fa = [i.as_posix() for i in f]

pd.DataFrame({
    "sample": samples,
    "taxon": genera,
    "genome": fa
    }).to_csv("<path to>/damage_inference_model_testing/genomes.csv", index=False)